# NEXUS temporary GPU image worker
Temporary SD-Turbo image-generation API for NEXUS. Image jobs only; not video/audio. Colab sessions can disconnect and the public tunnel URL changes on restart.

**Keep this notebook private. Never share the worker token or notebook output.**

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Runtime settings, then rerun.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install fastapi 'uvicorn[standard]' diffusers transformers accelerate safetensors
!mkdir -p /content/bin
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/bin/cloudflared
!chmod +x /content/bin/cloudflared
!/content/bin/cloudflared --version
print('Dependencies and tunnel client installed.')

In [ ]:
%%writefile free_gpu_worker.py
import os, uuid, threading, time
from pathlib import Path
from typing import Any
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
import torch
from diffusers import AutoPipelineForText2Image

TOKEN = os.environ.get('NEXUS_WORKER_TOKEN', '')
MODEL_ID = os.environ.get('NEXUS_IMAGE_MODEL', 'stabilityai/sd-turbo')
BASE_URL = os.environ.get('NEXUS_PUBLIC_URL', '').rstrip('/')
OUT = Path('/tmp/nexus-outputs')
OUT.mkdir(parents=True, exist_ok=True)
if len(TOKEN) < 32: raise RuntimeError('A strong NEXUS_WORKER_TOKEN must be set before startup.')
if not torch.cuda.is_available(): raise RuntimeError('No CUDA GPU is available.')
pipe = AutoPipelineForText2Image.from_pretrained(MODEL_ID, torch_dtype=torch.float16, variant='fp16')
pipe.to('cuda')
pipe.set_progress_bar_config(disable=True)
app = FastAPI(title='NEXUS Temporary GPU Image Worker')
jobs: dict[str, dict[str, Any]] = {}
lock = threading.Lock()
class JobRequest(BaseModel):
    engine: str = 'image'
    prompt: str = Field(min_length=1, max_length=12000)
    options: dict[str, Any] = Field(default_factory=dict)
def authorize(value: str | None):
    if value != f'Bearer {TOKEN}': raise HTTPException(status_code=401, detail='Invalid worker token.')
def run_job(job_id: str, prompt: str, options: dict[str, Any]):
    try:
        with lock: jobs[job_id].update(state='running', progress=5, message='Generating image')
        width = max(512, min(768, int(options.get('width', 512)) // 8 * 8))
        height = max(512, min(768, int(options.get('height', 512)) // 8 * 8))
        steps = max(1, min(4, int(options.get('steps', 2))))
        image = pipe(prompt=prompt, num_inference_steps=steps, guidance_scale=0.0, width=width, height=height).images[0]
        path = OUT / f'{job_id}.png'
        image.save(path)
        with lock: jobs[job_id].update(state='succeeded', progress=100, message='Image ready', output={'url': f'{BASE_URL}/outputs/{job_id}.png', 'mimeType': 'image/png'})
    except Exception as exc:
        with lock: jobs[job_id].update(state='failed', progress=0, message='Image generation failed', error=str(exc)[:400])
@app.get('/health')
def health(): return {'ok': True, 'gpu': torch.cuda.get_device_name(0), 'model': MODEL_ID}
@app.post('/jobs', status_code=202)
def submit(req: JobRequest, authorization: str | None = Header(default=None)):
    authorize(authorization)
    if req.engine != 'image': raise HTTPException(status_code=400, detail='This worker supports image jobs only.')
    job_id = uuid.uuid4().hex
    with lock: jobs[job_id] = {'id': job_id, 'engine': 'image', 'state': 'queued', 'progress': 0, 'message': 'Queued'}
    threading.Thread(target=run_job, args=(job_id, req.prompt, req.options), daemon=True).start()
    return jobs[job_id]
@app.get('/jobs/{job_id}')
def status(job_id: str, authorization: str | None = Header(default=None)):
    authorize(authorization)
    with lock:
        if job_id not in jobs: raise HTTPException(status_code=404, detail='Job not found; the temporary notebook may have restarted.')
        return dict(jobs[job_id])
@app.get('/outputs/{filename}')
def output(filename: str):
    if '/' in filename or not filename.endswith('.png'): raise HTTPException(status_code=404)
    path = OUT / filename
    if not path.is_file(): raise HTTPException(status_code=404)
    return FileResponse(path, media_type='image/png', filename=filename)


In [ ]:
import os, secrets, subprocess, time, re, requests, shutil
from pathlib import Path
cloudflared = shutil.which('cloudflared') or '/content/bin/cloudflared'
if not Path(cloudflared).is_file():
    raise FileNotFoundError('cloudflared is missing. Re-run the dependency installation cell above, then run this cell again.')
token = secrets.token_urlsafe(36)
tunnel_log = open('/tmp/nexus-cloudflared.log', 'w')
tunnel = subprocess.Popen([cloudflared, 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:7860'], stdout=tunnel_log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    log_text = Path('/tmp/nexus-cloudflared.log').read_text(errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        public_url = match.group(0)
        break
    if tunnel.poll() is not None:
        raise RuntimeError('Cloudflare tunnel stopped. Check /tmp/nexus-cloudflared.log.')
    time.sleep(2)
if not public_url:
    raise RuntimeError('Tunnel URL was not detected. Check /tmp/nexus-cloudflared.log.')
os.environ['NEXUS_WORKER_TOKEN'] = token
os.environ['NEXUS_PUBLIC_URL'] = public_url
server_log = open('/tmp/nexus-uvicorn.log', 'w')
server = subprocess.Popen(['uvicorn', 'free_gpu_worker:app', '--host', '127.0.0.1', '--port', '7860'], stdout=server_log, stderr=subprocess.STDOUT)
for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError('Worker stopped. Check /tmp/nexus-uvicorn.log before continuing.')
    try:
        if requests.get('http://127.0.0.1:7860/health', timeout=2).ok:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Worker did not become healthy within 6 minutes.')
for attempt in range(8):
    try:
        health = requests.get(public_url + '/health', timeout=30)
        health.raise_for_status()
        print('WORKER HEALTH:', health.json())
        break
    except Exception:
        if attempt == 7:
            raise
        time.sleep(5)
print('\\nCopy these into Vercel Preview environment variables:')
print('NEXUS_IMAGE_ENGINE_URL =', public_url)
print('NEXUS_IMAGE_ENGINE_TOKEN =', token)
print('\\nKeep this Colab session running. Treat the token as a password; do not paste it into ChatGPT or share screenshots containing it.')

## Connect NEXUS
In Vercel → Project → Settings → Environment Variables, add the two values printed by the launch cell as **Preview** variables: `NEXUS_IMAGE_ENGINE_URL` and `NEXUS_IMAGE_ENGINE_TOKEN`. Keep them private. A new Colab session creates a new URL and token, so update both after restarting. This notebook is an experimental image-only worker; it does not provide video or audio generation.